In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
import seaborn as sns
import matplotlib.pyplot as plt

In [25]:
# 1. Load Data
df = pd.read_csv('/content/prediction dataset.csv')
print("Original Data Shape:", df.shape)
print(df.head())

Original Data Shape: (100000, 9)
   gender   age  hypertension  heart_disease smoking_history    bmi  \
0  Female  80.0             0              1           never  25.19   
1  Female  54.0             0              0         No Info  27.32   
2    Male  28.0             0              0           never  27.32   
3  Female  36.0             0              0         current  23.45   
4    Male  76.0             1              1         current  20.14   

   HbA1c_level  blood_glucose_level  diabetes  
0          6.6                  140         0  
1          6.6                   80         0  
2          5.7                  158         0  
3          5.0                  155         0  
4          4.8                  155         0  


In [26]:
# 2. Basic Info

print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())


Data Types:
 gender                  object
age                    float64
hypertension             int64
heart_disease            int64
smoking_history         object
bmi                    float64
HbA1c_level            float64
blood_glucose_level      int64
diabetes                 int64
dtype: object

Missing Values:
 gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64


In [27]:
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns
target_col = 'diabetes'  # Define target_col early for consistent use

# Numerical: Replace with mean, only if there are missing values
for col in num_cols:
    if df[col].isnull().any():
        num_imputer = SimpleImputer(strategy='mean')
        df[col] = num_imputer.fit_transform(df[[col]])

# Categorical: Replace with mode, only if there are missing values
for col in cat_cols:
    if df[col].isnull().any():
        cat_imputer = SimpleImputer(strategy='most_frequent')
        df[col] = cat_imputer.fit_transform(df[[col]])

# Check missing values after handling
print(df.isnull().sum())

gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64


In [28]:
# 4. Encode Categorical Variables
le = LabelEncoder()

for col in cat_cols:
    if df[col].nunique() == 2:
        df[col] = le.fit_transform(df[col])

# One-hot Encoding (for more than 2 categories)
df = pd.get_dummies(
    df,
    columns=[col for col in cat_cols if df[col].nunique() > 2],
    drop_first=True
)

In [29]:
# 5. Outlier Detection and Treatment (using IQR)

import numpy as np

# Re-identify numerical columns in the current DataFrame suitable for outlier treatment
# Exclude the target column from outlier treatment if it's numerical
numerical_cols_for_outliers = [
    col for col in df.select_dtypes(include=['int64', 'float64']).columns
    if col != target_col
]

for col in numerical_cols_for_outliers:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] < lower, lower, df[col])

    df[col] = np.where(df[col] > upper, upper, df[col])

# Display dataset after outlier treatment

print(df.head())

    age  hypertension  heart_disease    bmi  HbA1c_level  blood_glucose_level  \
0  80.0           0.0            0.0  25.19          6.6                140.0   
1  54.0           0.0            0.0  27.32          6.6                 80.0   
2  28.0           0.0            0.0  27.32          5.7                158.0   
3  36.0           0.0            0.0  23.45          5.0                155.0   
4  76.0           0.0            0.0  20.14          4.8                155.0   

   diabetes  gender_Male  gender_Other  smoking_history_current  \
0         0        False         False                    False   
1         0        False         False                    False   
2         0         True         False                    False   
3         0        False         False                     True   
4         0         True         False                     True   

   smoking_history_ever  smoking_history_former  smoking_history_never  \
0                 False             

In [31]:
# 6. Feature Scaling

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df[numerical_cols_for_outliers] = scaler.fit_transform(df[numerical_cols_for_outliers])


In [32]:
# 7. Feature Selection (if classification target is available)
target_col = 'diabetes'
if target_col in df.columns:
   # Features and target
    X = df.drop(target_col, axis=1)
    y = df[target_col]
    bestfeatures = SelectKBest(score_func=f_classif, k=10)
    fit = bestfeatures.fit(X, y)
    selected_features = X.columns[fit.get_support()]
    df = df[selected_features.to_list() + [target_col]]
    print("\nSelected Features:", selected_features.to_list())


Selected Features: ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'gender_Male', 'smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current']


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [1 2] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


In [33]:
X = np.asarray(df[selected_features])
y = np.asarray(df[target_col])

X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size = 0.3, random_state = 4)

print ('Train set:', X_train.shape, y_train.shape)
print ('Test set:', X_test.shape, y_test.shape)

Train set: (70000, 10) (70000,)
Test set: (30000, 10) (30000,)


In [34]:
df.columns


Index(['age', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'gender_Male',
       'smoking_history_current', 'smoking_history_ever',
       'smoking_history_former', 'smoking_history_never',
       'smoking_history_not current', 'diabetes'],
      dtype='object')

In [35]:
# 1. Linear Regression
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.3144747897140855


In [36]:
# 2. Decision Tree
from sklearn.tree import DecisionTreeRegressor
model = DecisionTreeRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.37536557412628424


In [37]:
# 3. Random Forest
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.6736101781047519


In [38]:
# 4. K-Nearest Neighbors
from sklearn.neighbors import KNeighborsRegressor
model = KNeighborsRegressor(n_neighbors=5)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.6002744086220035


In [39]:
# 5. Support Vector Regression
from sklearn.svm import SVR
model = SVR()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.6086943222246026


In [40]:
# 6. Gradient Boosting
from sklearn.ensemble import GradientBoostingRegressor
model = GradientBoostingRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.7068530690508094


In [41]:
# 7. Ridge Regression
from sklearn.linear_model import Ridge
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.31447474338241077
